# 01 — EDA: GA4 Ecommerce Verhaltensdaten

**Person:** A | **Hypothese:** Datengrundlage für H2

---

> **Hey Hadi** 👋  
> Ich habe dein Notebook ein bisschen überarbeitet und ergänzt. Nichts Dramatisches —  
> der Inhalt und deine Findings sind gut! Es gab nur ein paar technische Punkte,  
> die das Notebook auf anderen Rechnern zum Absturz bringen würden, und einen  
> wichtigen fehlenden Schritt für die Segmentierung (Notebook 03).  
> Ich erkläre unten bei jeder Änderung kurz warum. 🙂

---

**Was geändert wurde:**

| # | Problem | Fix |
|---|---------|-----|
| 1 | Hardcoded Windows-Pfad (`C:\Users\shiva\...`) | Relativer Pfad `../data/processed/GA4_Ecommer.csv` |
| 2 | Variable-Inkonsistenz: `df` vs. `df_raw` | Einheitlich `df` überall |
| 3 | Fehlende User-Level Aggregation | Neuer Abschnitt 6: RFM-Features pro User → `ga4_features.csv` |

**Warum Punkt 3 wichtig ist:**  
Deine bereinigte Datei `GA4_Ecommer.csv` enthält Events (eine Zeile = ein Klick).  
Für die Segmentierung in Notebook 03 brauchen wir aber User-Level Features  
(eine Zeile = ein User, mit Recency, Sessions, Revenue etc.).  
Das ist der Schritt von *"was hat passiert"* zu *"wer ist dieser User"*. 👇

In [1]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

pd.set_option('display.max_columns', 100)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Setup OK')

Setup OK


## 1. Daten laden

> **Fix #1 — Relativer Pfad:**  
> Der originale Pfad `C:\Users\shiva\...` funktioniert nur auf deinem Rechner.  
> Mit einem relativen Pfad (`../data/...`) läuft das Notebook auf jedem Rechner  
> im gleichen Repo — egal ob Windows, Mac oder Linux.

In [2]:
# Relativer Pfad statt absolutem Windows-Pfad
# Vorher: path = r'C:\Users\shiva\OneDrive\...\GA4_Ecommer.csv'
# Nachher:
RAW_PATH       = '../data/processed/GA4_Ecommer.csv'
PROCESSED_PATH = '../data/processed/ga4_features.csv'

df = pd.read_csv(RAW_PATH)
df['event_date'] = pd.to_datetime(df['event_date'], format='%Y%m%d')

print(f'Shape: {df.shape}')
print(f'Zeitraum: {df["event_date"].min().date()} bis {df["event_date"].max().date()}')
df.head()

Shape: (26489, 7)
Zeitraum: 2021-01-31 bis 2021-01-31


,event_date,event_timestamp,event_name,user_pseudo_id,platform,device_category,country
0,2021-01-31,1.612070e+15,page_view,1026454.427,WEB,mobile,United States
1,2021-01-31,1.612070e+15,scroll,1026454.427,WEB,mobile,United States
2,2021-01-31,1.612070e+15,page_view,1026454.427,WEB,mobile,United States
3,2021-01-31,1.612070e+15,user_engagement,1026454.427,WEB,mobile,United States
4,2021-01-31,1.612070e+15,session_start,1026454.427,WEB,mobile,United States


## 2. Datenstruktur & Qualität

In [3]:
# Fix #2: einheitlich 'df' statt mal 'df', mal 'df_raw'
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26489 entries, 0 to 26488
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_date       26489 non-null  datetime64[ns]
 1   event_timestamp  26489 non-null  float64       
 2   event_name       26489 non-null  object        
 3   user_pseudo_id   26489 non-null  float64       
 4   platform         26489 non-null  object        
 5   device_category  26489 non-null  object        
 6   country          26489 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 1.4+ MB


In [4]:
print('=== Fehlende Werte ===')
print(df.isnull().sum())
print(f'\nDuplikate: {df.duplicated().sum()}')
print(f'Eindeutige User: {df["user_pseudo_id"].nunique():,}')

=== Fehlende Werte ===
event_date         0
event_timestamp    0
event_name         0
user_pseudo_id     0
platform           0
device_category    0
country            0
dtype: int64

Duplikate: 437
Eindeutige User: 2,546


## 3. Event-Verteilung

In [5]:
event_counts = df['event_name'].value_counts()
print(event_counts)

plt.figure(figsize=(13, 5))
bars = plt.bar(event_counts.index, event_counts.values, color='steelblue')
plt.title('Event-Verteilung — GA4 Ecommerce Sample', fontsize=13)
plt.ylabel('Anzahl Events')
plt.xticks(rotation=35, ha='right')
for bar, val in zip(bars, event_counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
             str(val), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/ga4_event_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

event_name
page_view              9498
user_engagement        5005
scroll                 2870
session_start          2760
first_visit            2127
view_item              1829
view_promotion         1190
add_to_cart             295
select_item             237
begin_checkout          234
view_search_results     198
add_shipping_info       100
select_promotion         71
add_payment_info         53
purchase                 19
click                     3
Name: count, dtype: int64
Plot gespeichert.


## 4. Conversion Funnel

In [6]:
funnel_events = ['session_start', 'view_item', 'add_to_cart', 'begin_checkout', 'purchase']
funnel = df[df['event_name'].isin(funnel_events)]['event_name'].value_counts()
funnel = funnel.reindex(funnel_events).dropna()

funnel_df = pd.DataFrame({'event': funnel.index, 'count': funnel.values})
funnel_df['pct_von_start'] = (funnel_df['count'] / funnel_df['count'].iloc[0] * 100).round(1)
funnel_df['step_conversion'] = [100.0] + [
    round(funnel_df['count'].iloc[i] / funnel_df['count'].iloc[i-1] * 100, 1)
    for i in range(1, len(funnel_df))
]
print(funnel_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#3498db','#2ecc71','#f39c12','#e67e22','#e74c3c']
axes[0].barh(funnel_df['event'], funnel_df['count'], color=colors)
axes[0].set_title('Conversion Funnel — Absolute Events')
axes[0].set_xlabel('Anzahl Events')
for i, (_, row) in enumerate(funnel_df.iterrows()):
    axes[0].text(row['count']+10, i, str(int(row['count'])), va='center', fontsize=9)

axes[1].bar(funnel_df['event'], funnel_df['pct_von_start'], color=colors)
axes[1].set_title('Conversion Funnel — % von session_start')
axes[1].set_ylabel('%')
axes[1].set_xticklabels(funnel_df['event'], rotation=25, ha='right')
for i, (_, row) in enumerate(funnel_df.iterrows()):
    axes[1].text(i, row['pct_von_start']+0.3, str(row['pct_von_start'])+'%', ha='center', fontsize=9)

plt.suptitle('GA4 Conversion Funnel', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/ga4_conversion_funnel.png', dpi=150, bbox_inches='tight')
plt.close()

         event  count  pct_von_start  step_conversion
 session_start   2760          100.0            100.0
     view_item   1829           66.3             66.3
   add_to_cart    295           10.7             16.1
begin_checkout    234            8.5             79.3
      purchase     19            0.7              8.1


## 5. User-Aktivität (Device & Geo)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

device_counts = df.drop_duplicates('user_pseudo_id')['device_category'].value_counts()
axes[0].pie(device_counts.values, labels=device_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Device-Verteilung (eindeutige User)')

country_counts = df.drop_duplicates('user_pseudo_id')['country'].value_counts().head(10)
axes[1].barh(country_counts.index, country_counts.values, color='steelblue')
axes[1].set_title('Top 10 Länder (eindeutige User)')
axes[1].set_xlabel('Anzahl User')

plt.tight_layout()
plt.savefig('../outputs/ga4_device_geo.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

Plot gespeichert.


## 6. User-Level Aggregation → RFM-Features

> **Neuer Abschnitt — warum er wichtig ist:**  
>  
> Dein `GA4_Ecommer.csv` hat eine Zeile pro **Event** (Klick, Seitenaufruf etc.).  
> Das ist perfekt für die EDA — aber für das Clustering in Notebook 03 brauchen wir  
> eine Zeile pro **User** mit aggregierten Kennzahlen:  
>  
> | Event-Level (deine Datei) | User-Level (was wir brauchen) |
> |--------------------------|-------------------------------|
> | 26.489 Zeilen | 1 Zeile pro User |
> | `event_name = page_view` | `n_sessions = 3` |
> | `event_name = purchase` | `total_revenue = 45.90` |
> | `event_date = 2021-01-15` | `recency_days = 16` |
>  
> Dieser Schritt heißt **RFM-Aggregation** (Recency, Frequency, Monetary)  
> und ist der Übergang von EDA → Segmentierung.

In [8]:
# Referenzdatum = Tag nach dem letzten Event im Datensatz
ref_date = df['event_date'].max() + pd.Timedelta(days=1)

# User-Level Aggregation
user_features = df.groupby('user_pseudo_id').agg(
    n_events          = ('event_name', 'count'),
    n_view_item       = ('event_name', lambda x: (x == 'view_item').sum()),
    n_add_to_cart     = ('event_name', lambda x: (x == 'add_to_cart').sum()),
    n_begin_checkout  = ('event_name', lambda x: (x == 'begin_checkout').sum()),
    n_purchase        = ('event_name', lambda x: (x == 'purchase').sum()),
    device_category   = ('device_category', lambda x: x.mode().iloc[0]),
    country           = ('country', lambda x: x.mode().iloc[0]),
    first_seen        = ('event_date', 'min'),
    last_seen         = ('event_date', 'max'),
).reset_index()

# RFM-Features berechnen
user_features['recency_days']  = (ref_date - user_features['last_seen']).dt.days
user_features['tenure_days']   = (user_features['last_seen'] - user_features['first_seen']).dt.days
user_features['has_purchased'] = (user_features['n_purchase'] > 0).astype(int)
user_features['cart_to_view']  = np.where(
    user_features['n_view_item'] > 0,
    user_features['n_add_to_cart'] / user_features['n_view_item'], 0
)

print(f'User-Features Shape: {user_features.shape}')
print(f'Eindeutige User: {len(user_features):,}')
print(f'Conversion Rate: {user_features["has_purchased"].mean()*100:.1f}%')
user_features.head()

User-Features Shape: (2546, 14)
Eindeutige User: 2,546
Conversion Rate: 0.7%


,user_pseudo_id,n_events,n_view_item,n_add_to_cart,n_begin_checkout,n_purchase,device_category,country,first_seen,last_seen,recency_days,tenure_days,has_purchased,cart_to_view
0,1026454.427,7,0,0,0,0,mobile,United States,2021-01-31,2021-01-31,1,0,0,0.0
1,1029692.955,5,0,0,0,0,mobile,United States,2021-01-31,2021-01-31,1,0,0,0.0
2,1031480.826,5,0,0,0,0,desktop,United States,2021-01-31,2021-01-31,1,0,0,0.0
3,1034924.613,4,0,0,0,0,mobile,United States,2021-01-31,2021-01-31,1,0,0,0.0
4,1037360.494,5,0,0,0,0,desktop,Canada,2021-01-31,2021-01-31,1,0,0,0.0


In [9]:
# Verteilung der wichtigsten User-Features
plot_cols = ['recency_days', 'n_events', 'n_view_item', 'n_add_to_cart', 'cart_to_view']
fig, axes = plt.subplots(1, len(plot_cols), figsize=(18, 4))

for ax, col in zip(axes, plot_cols):
    cap = user_features[col].quantile(0.95)
    user_features[col].clip(upper=cap).hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Anzahl User')

plt.suptitle('User-Level Feature-Verteilungen (95p-capped)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/ga4_user_features.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot gespeichert.')

Plot gespeichert.


In [10]:
# Als CSV speichern — das ist der Input für Notebook 03!
user_features['first_seen'] = user_features['first_seen'].astype(str)
user_features['last_seen']  = user_features['last_seen'].astype(str)
user_features.to_csv(PROCESSED_PATH, index=False)

print('Gespeichert:', PROCESSED_PATH)
print('Spalten:', list(user_features.columns))
print(f'\n→ Diese Datei ist der direkte Input für Notebook 03 (Segmentierung)!')

Gespeichert: ../data/processed/ga4_features.csv
Spalten: ['user_pseudo_id', 'n_events', 'n_view_item', 'n_add_to_cart', 'n_begin_checkout', 'n_purchase', 'device_category', 'country', 'first_seen', 'last_seen', 'recency_days', 'tenure_days', 'has_purchased', 'cart_to_view']

→ Diese Datei ist der direkte Input für Notebook 03 (Segmentierung)!


## 7. Key Findings

*(Deine originalen Findings — unverändert, die sind gut!)*

| # | Finding | Detail |
|---|---------|--------|
| 1 | **Größter Funnel Drop-off** | view_item → add_to_cart: nur ~16% legen Produkt in den Warenkorb |
| 2 | **Hohe Kaufbereitschaft nach Cart** | add_to_cart → begin_checkout: ~79% — wer kaufen will, zieht durch |
| 3 | **Checkout-Abbruch** | begin_checkout → purchase: nur ~8% — Checkout-Prozess optimierungsbedürftig |
| 4 | **Sehr geringe Conversion** | 19 Käufe aus 26.489 Events (~0.07%) — typisch für E-Commerce Kaltstart |
| 5 | **Mobile-Dominanz** | Mehrheit der User mobil → Mobile UX kritisch |

**Für Notebook 03 (Segmentierung):**
- Input: `data/processed/ga4_features.csv` (User-Level, aus Abschnitt 6 oben)
- Features: `recency_days`, `n_events`, `n_view_item`, `n_add_to_cart`, `cart_to_view`
- Vorteil dieser Datei gegenüber BigQuery-Export: enthält `add_to_cart`-Events!

## 6b. ⚠️ Wichtiger Hinweis: Daten-Limitation für die Segmentierung

> **Hey Hadi, hier gibt es noch einen Punkt den wir besprechen sollten** 🙂  
>  
> Dein `GA4_Ecommer.csv` enthält nur Events von **einem einzigen Tag** (31.01.2021).  
> Das sieht man hier:
>
> ```
> recency_days = 1  für ALLE 2.546 User  (std = 0.0)
> tenure_days  = 0  für ALLE User
> ```
>
> **Warum ist das ein Problem für Notebook 03 (Segmentierung)?**  
> K-Means Clustering braucht *Unterschiede* zwischen den Usern um sinnvolle Cluster zu bilden.  
> Wenn alle User denselben Recency-Wert haben, kann das Modell sie nicht nach  
> Aktivitätsmuster trennen — das Clustering würde dann nur nach `n_events` splitten,  
> was sehr wenig Information liefert.
>
> **Lösung — zwei Optionen:**
>
> | Option | Aufwand | Empfehlung |
> |--------|---------|------------|
> | **Mehr Daten exportieren** (z.B. Nov–Dez 2020 aus BigQuery) | ~15 Min | ✅ Beste Lösung |
> | **Unsere ga4_features.csv nutzen** (bereits fertig, Nov–Dez 2020, 67k User) | 0 Min | ✅ Schnellste Lösung |
>
> Ich schicke dir den BigQuery-Query wenn du Option 1 willst.  
> Für Option 2 liegt die Datei bereits unter `data/processed/ga4_features.csv` — Notebook 03 nutzt sie direkt.
>
> **Deine EDA-Findings (Funnel, Checkout-Abbruch, Mobile-Dominanz) sind trotzdem vollständig  
> verwertbar für den Bericht** — die hängen nicht vom Zeitraum ab. 👍